# Deep Learning Hybrid Pipeline (BERT + RoBERTa)
This notebook replicates the deep learning pipeline from the paper, featuring a Hybrid model that fuses features from both BERT and RoBERTa.

In [ ]:
import pandas as pd
import numpy as np
import time
import os
from collections import Counter
from tqdm.auto import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, RobertaTokenizer
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score

from augmentation import RequirementAugmenter
from models import HybridBERT_RoBERTa

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


## 1. Load Data & Encode Labels

In [ ]:
data_dir = '../data/processed'
train_df = pd.read_csv(f"{data_dir}/train.csv")
val_df = pd.read_csv(f"{data_dir}/val.csv")
test_df = pd.read_csv(f"{data_dir}/test.csv")

# We will focus on Multi-class NFR classification as the primary baseline evaluation.
# Filter NFRs
train_nfr = train_df[train_df['label_binary'] == 'NFR'].copy().reset_index(drop=True)
val_nfr = val_df[val_df['label_binary'] == 'NFR'].copy().reset_index(drop=True)
test_nfr = test_df[test_df['label_binary'] == 'NFR'].copy().reset_index(drop=True)

# Map labels to integers
unique_labels = sorted(train_nfr['sub_NFR'].unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}
num_classes = len(unique_labels)

train_nfr['label_id'] = train_nfr['sub_NFR'].map(label2id)
val_nfr['label_id'] = val_nfr['sub_NFR'].map(label2id)
test_nfr['label_id'] = test_nfr['sub_NFR'].map(label2id)

print(f"Number of classes: {num_classes}")
print("Label mapping:", label2id)


## 2. Apply Data Augmentation to Train Set
Like the SVM notebook, we apply the 5 semantics-preserving augmentations to balance minority NFR classes.

In [ ]:
aug = RequirementAugmenter()
class_counts = Counter(train_nfr['sub_NFR'])
max_count = max(class_counts.values())

augmented_rows = []
for label, count in class_counts.items():
    if count < max_count:
        num_to_generate = max_count - count
        existing_texts = train_nfr[train_nfr['sub_NFR'] == label]['text'].tolist()
        for _ in range(num_to_generate):
            import random
            base_text = random.choice(existing_texts)
            try:
                new_text = aug.augment(base_text, strategy='random')
            except:
                new_text = base_text
            augmented_rows.append({'text': new_text, 'sub_NFR': label, 'label_id': label2id[label]})

aug_df = pd.DataFrame(augmented_rows)
train_nfr_balanced = pd.concat([train_nfr, aug_df], ignore_index=True)
print("Train distribution after augmentation:", dict(Counter(train_nfr_balanced['sub_NFR'])))


## 3. Custom Dataset with 2 Tokenizers

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class HybridRequirementDataset(Dataset):
    def __init__(self, texts, labels, bert_tokenizer, roberta_tokenizer, max_length=32):
        self.texts = texts
        self.labels = labels
        self.bert_tokenizer = bert_tokenizer
        self.roberta_tokenizer = roberta_tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        bert_enc = self.bert_tokenizer(
            text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
        )
        roberta_enc = self.roberta_tokenizer(
            text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
        )
        
        return {
            'input_ids_bert': bert_enc['input_ids'].squeeze(0),
            'attention_mask_bert': bert_enc['attention_mask'].squeeze(0),
            'input_ids_roberta': roberta_enc['input_ids'].squeeze(0),
            'attention_mask_roberta': roberta_enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

batch_size = 16 # Effective batch size will be 32 via gradient accumulation

train_dataset = HybridRequirementDataset(train_nfr_balanced['text'].values, train_nfr_balanced['label_id'].values, bert_tokenizer, roberta_tokenizer)
val_dataset = HybridRequirementDataset(val_nfr['text'].values, val_nfr['label_id'].values, bert_tokenizer, roberta_tokenizer)
test_dataset = HybridRequirementDataset(test_nfr['text'].values, test_nfr['label_id'].values, bert_tokenizer, roberta_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


## 4. Model Training

In [ ]:
model = HybridBERT_RoBERTa(num_classes=num_classes)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.02)
criterion = torch.nn.CrossEntropyLoss()
# scaler removed for CPU # For Mixed Precision FP16

epochs = 3 # Feel free to increase to 30, but 10 is enough for a demo
accumulation_steps = 1
best_val_f1 = 0.0
patience = 2
patience_counter = 0

os.makedirs('../models/hybrid', exist_ok=True)
best_model_path = '../models/hybrid/hybrid_multiclass.pt'

print("Starting training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} (Train)")):
        input_ids_bert = batch['input_ids_bert'].to(device)
        mask_bert = batch['attention_mask_bert'].to(device)
        input_ids_rob = batch['input_ids_roberta'].to(device)
        mask_rob = batch['attention_mask_roberta'].to(device)
        labels = batch['labels'].to(device)
        
        if True: # autocast removed for CPU
            logits = model(input_ids_bert, mask_bert, input_ids_rob, mask_rob)
            loss = criterion(logits, labels) / accumulation_steps
            
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            
            optimizer.zero_grad()
            
        total_loss += loss.item() * accumulation_steps
        
    # Validation
    model.eval()
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            if True: # autocast removed for CPU
                logits = model(
                    batch['input_ids_bert'].to(device), batch['attention_mask_bert'].to(device),
                    batch['input_ids_roberta'].to(device), batch['attention_mask_roberta'].to(device)
                )
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(batch['labels'].numpy())
            
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f} | Val Macro F1: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        patience_counter = 0
        print(f"-> Saved new best model (F1={val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break


## 5. Evaluation on Test Set

In [ ]:
# Load the best model
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_labels = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        if True: # autocast removed for CPU
            logits = model(
                batch['input_ids_bert'].to(device), batch['attention_mask_bert'].to(device),
                batch['input_ids_roberta'].to(device), batch['attention_mask_roberta'].to(device)
            )
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_labels.extend(batch['labels'].numpy())

print("\n--- Hybrid Model Test Results ---")
print(f"Accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=[id2label[i] for i in range(num_classes)]))
